# I-S40 → I-S100 SegResNet 연속 실험

팀원의 I-S00~I-S30은 reference로만 유지한다. 이 노트북 하나에서 신규 7개 실험을 순서대로 실행한다. 모두 300 epochs, early stopping 없음, Top-3 periodic anchor의 ±5 epoch full-volume 재평가, best 확률맵 저장 및 threshold sweep을 사용한다.

In [1]:
from pathlib import Path
import json, os, signal, subprocess, time
import pandas as pd
from IPython.display import display

REPO = Path("/home/stu03/workspace/model")
PYTHON = REPO / ".venv/bin/python"
assert PYTHON.is_file(), PYTHON

EXPERIMENTS = ["I-S40","I-S50","I-S60","I-S70","I-S80","I-S90","I-S100"]
CONFIGS = {x: REPO / f"configs/experiments/ischemia/{x}.yaml" for x in EXPERIMENTS}
CONFIGS

{'I-S40': PosixPath('/home/stu03/workspace/model/configs/experiments/ischemia/I-S40.yaml'),
 'I-S50': PosixPath('/home/stu03/workspace/model/configs/experiments/ischemia/I-S50.yaml'),
 'I-S60': PosixPath('/home/stu03/workspace/model/configs/experiments/ischemia/I-S60.yaml'),
 'I-S70': PosixPath('/home/stu03/workspace/model/configs/experiments/ischemia/I-S70.yaml'),
 'I-S80': PosixPath('/home/stu03/workspace/model/configs/experiments/ischemia/I-S80.yaml'),
 'I-S90': PosixPath('/home/stu03/workspace/model/configs/experiments/ischemia/I-S90.yaml'),
 'I-S100': PosixPath('/home/stu03/workspace/model/configs/experiments/ischemia/I-S100.yaml')}

## 실험 구성

| ID | 핵심 변경 |
|---|---|
| I-S40 | Dice-Focal + Deep Supervision |
| I-S50 | DiceCE + Deep Supervision |
| I-S60 | Tversky (FN 강화) + Deep Supervision |
| I-S70 | Generalized Dice-Focal + Deep Supervision |
| I-S80 | Dice-Focal + Foreground Sampling |
| I-S90 | DWI/ADC Dual-Stem SegResNet |
| I-S100 | 독립 Dual-Encoder + scale별 gated feature fusion + shared decoder |

In [2]:
# 실제 학습 없이 7개 명령/경로 검사
subprocess.run([str(PYTHON), str(REPO / "scripts/ischemia/training/run_i_s_sequence.py"), "--dry-run"], cwd=REPO, check=True)

[START] I-S40
{
  "experiment": "I-S40",
  "mode": "resume",
  "data_root": "/home/stu03/workspace/datasets/ischemic/SegResNet",
  "results": "/home/stu03/workspace/model/runs/I-segresnet/I-S40",
  "command": [
    "/home/stu03/workspace/model/.venv/bin/python",
    "-m",
    "brainon_ai.models.ischemia.segresnet_hy.train",
    "--config",
    "/home/stu03/workspace/model/configs/experiments/ischemia/I-S40.yaml",
    "--data-root",
    "/home/stu03/workspace/datasets/ischemic/SegResNet",
    "--backup-root",
    "/home/stu03/workspace/model/runs/I-segresnet/I-S40"
  ]
}
[DONE] I-S40
[START] I-S50
{
  "experiment": "I-S50",
  "mode": "new",
  "data_root": "/home/stu03/workspace/datasets/ischemic/SegResNet",
  "results": "/home/stu03/workspace/model/runs/I-segresnet/I-S50",
  "command": [
    "/home/stu03/workspace/model/.venv/bin/python",
    "-m",
    "brainon_ai.models.ischemia.segresnet_hy.train",
    "--config",
    "/home/stu03/workspace/model/configs/experiments/ischemia/I-S50.yam

CompletedProcess(args=['/home/stu03/workspace/model/.venv/bin/python', '/home/stu03/workspace/model/scripts/ischemia/training/run_i_s_sequence.py', '--dry-run'], returncode=0)

## 연속 학습 시작/재개

완료된 실험은 건너뛰고, 미완료 실험은 `checkpoint_latest.pth`에서 재개한다. 한 실험의 최종 neighborhood validation과 확률맵 저장이 끝난 뒤 다음 실험으로 이동한다.

In [3]:
LOG = REPO / "runs/I-segresnet/I-S40_S100_sequence.log"
PID_FILE = REPO / "runs/I-segresnet/I-S40_S100_sequence.pid"
PID_FILE.parent.mkdir(parents=True, exist_ok=True)
if PID_FILE.is_file():
    old_pid=int(PID_FILE.read_text())
    try: os.kill(old_pid,0); raise RuntimeError(f"이미 실행 중입니다: PID={old_pid}")
    except ProcessLookupError: pass
handle=LOG.open("a",buffering=1)
proc=subprocess.Popen([str(PYTHON),str(REPO/"scripts/ischemia/training/run_i_s_sequence.py")],cwd=REPO,stdout=handle,stderr=subprocess.STDOUT,start_new_session=True)
PID_FILE.write_text(str(proc.pid)); print(f"started PID={proc.pid} | log={LOG}")

started PID=2197362 | log=/home/stu03/workspace/model/runs/I-segresnet/I-S40_S100_sequence.log


## 실시간 진행률

In [5]:
from IPython.display import clear_output

def process_alive(pid_file):
    if not pid_file.is_file(): return False
    try:
        os.kill(int(pid_file.read_text()), 0); return True
    except (ProcessLookupError, ValueError): return False

try:
    while True:
        rows=[]
        for eid in EXPERIMENTS:
            run=REPO/f"runs/I-segresnet/{eid}"; p=run/"progress.json"
            row={"experiment":eid,"status":"waiting","result":str(run)}
            if p.is_file(): row.update(json.loads(p.read_text()))
            row["probability_maps"]=len(list((run/"final_metrics/val_probabilities").glob("*.npz")))
            rows.append(row)
        clear_output(wait=True)
        print(time.strftime("Updated: %Y-%m-%d %H:%M:%S"))
        display(pd.DataFrame(rows).reindex(columns=["experiment","status","current_epoch","total_epochs","progress_percent","eta_seconds","best_full_macro_dice","selected_epoch","probability_maps","result"]))
        if LOG.is_file(): print("\n--- latest log ---\n"+"".join(LOG.read_text(errors="replace").splitlines(True)[-20:]))
        if not process_alive(PID_FILE):
            print("\n연속 실행 프로세스가 종료되었습니다. 마지막 상태입니다."); break
        time.sleep(10)
except KeyboardInterrupt:
    print("모니터만 중단했습니다. background 학습은 계속됩니다.")

,experiment,status,current_epoch,total_epochs,progress_percent,eta_seconds,best_full_macro_dice,selected_epoch,probability_maps,result
0,I-S40,completed,300.0,300.0,100.00,0.0,0.761408,274.0,38,/home/stu03/workspace/model/runs/I-segresnet/I...
1,I-S50,running,2.0,300.0,0.67,2187.0,NaN,NaN,0,/home/stu03/workspace/model/runs/I-segresnet/I...
2,I-S60,waiting,NaN,NaN,NaN,NaN,NaN,NaN,0,/home/stu03/workspace/model/runs/I-segresnet/I...
3,I-S70,waiting,NaN,NaN,NaN,NaN,NaN,NaN,0,/home/stu03/workspace/model/runs/I-segresnet/I...
4,I-S80,waiting,NaN,NaN,NaN,NaN,NaN,NaN,0,/home/stu03/workspace/model/runs/I-segresnet/I...
5,I-S90,waiting,NaN,NaN,NaN,NaN,NaN,NaN,0,/home/stu03/workspace/model/runs/I-segresnet/I...
6,I-S100,waiting,NaN,NaN,NaN,NaN,NaN,NaN,0,/home/stu03/workspace/model/runs/I-segresnet/I...


Test : 37
입력 : DWI + ADC (2 channels)
Patch: (80, 96, 80)
/home/stu03/workspace/model/src/brainon_ai/models/ischemia/segresnet_hy/train.py:116: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=amp)
/home/stu03/workspace/model/src/brainon_ai/models/ischemia/segresnet_hy/train.py:176: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=amp):
/home/stu03/workspace/model/.venv/lib/python3.12/site-packages/monai/transforms/utils.py:682: UserWarning: Num foregrounds 0, Num backgrounds 1003520, unable to generate class balanced samples, setting `pos_ratio` to 0.
  warnings.warn(

Epoch 1/300 완료
Train Loss      : 0.7617
Full Val Dice   : scheduled full-volume validation
Best Full Dice  : -
Early Stopping  : disabled (fixed 300 epochs)
Learning Rate   : 0.00099997
E

## 중단 (필요할 때만)

In [ ]:
pid=int(PID_FILE.read_text()); os.killpg(pid,signal.SIGTERM); print(f"stop requested: {pid}")

## 단일 모델 결과 비교

필수 지표: positive-case Macro Dice, voxel recall/precision, lesion recall/precision/F1/F2, small-lesion Dice/Recall, FP lesions/scan, negative-case FP rate.

In [ ]:
summary=[]
for eid in EXPERIMENTS:
    run=REPO/f"runs/I-segresnet/{eid}"; sweep=run/"final_metrics/threshold_sweep.csv"; sel=run/"final_selection.json"
    if not sweep.is_file(): summary.append({"experiment":eid,"status":"not_ready"}); continue
    df=pd.read_csv(sweep); rec=df.sort_values(["lesion_f2","macro_dice_positive"],ascending=False).iloc[0].to_dict(); rec.update(experiment=eid,status="complete",selected_epoch=(json.loads(sel.read_text())["selected"]["epoch"] if sel.is_file() else None)); summary.append(rec)
display(pd.DataFrame(summary))

## 최종 soft-voting

단일 성능과 다양성을 확인한 뒤 `SELECTED`에 2~4개 후보를 넣는다. 확률맵을 평균하고 threshold 0.10~0.90을 탐색해 FP/scan 제한 안에서 lesion F2가 가장 높은 값을 추천한다. nnU-Net과 SegResNet의 case/shape가 동일한지도 실행 시 강제 검증한다.

In [ ]:
# 예시 경로. 실제 완료 결과 중 Dice/recall/small-lesion/다양성이 보완적인 모델로 수정하세요.
NN_ROOT = REPO / "runs/I-nnunet/Dataset502_ISLES2022"
CANDIDATES = {
 "I-N30": NN_ROOT/"nnUNetTrainerIschemiaInstanceCenter__IschemiaPlansResidualEncoderICIV2__3d_fullres/fold_0/best_probability_validation",
 "I-N40": NN_ROOT/"nnUNetTrainerIschemiaComponentSampling__IschemiaPlansResidualEncoderComponentSampling__3d_fullres/fold_0/best_probability_validation",
 **{eid: REPO/f"runs/I-segresnet/{eid}/final_metrics/val_probabilities" for eid in EXPERIMENTS},
}
SELECTED = ["I-N30", "I-N40", "I-S40"]
{k:str(CANDIDATES[k]) for k in SELECTED}

In [ ]:
missing=[f"{k}: {CANDIDATES[k]}" for k in SELECTED if not CANDIDATES[k].is_dir()]
if missing: raise FileNotFoundError("확률맵이 아직 없습니다:\n"+"\n".join(missing))
cmd=[str(PYTHON),str(REPO/"scripts/ischemia/ensemble/evaluate_soft_voting.py"),"--label-dir","/home/stu03/workspace/datasets/ischemic/SegResNet/val/labels","--output-dir",str(REPO/"runs/I-ensemble/final_soft_voting")]
for k in SELECTED: cmd += ["--model",f"{k}={CANDIDATES[k]}"]
subprocess.run(cmd,cwd=REPO,check=True)
display(pd.read_csv(REPO/"runs/I-ensemble/final_soft_voting/ensemble_threshold_sweep.csv").sort_values(["candidate","lesion_f2"],ascending=[True,False]).groupby("candidate").head(1))
display(pd.read_csv(REPO/"runs/I-ensemble/final_soft_voting/model_diversity.csv"))